# Linear Algebra in Machine Learning
**Topic:** Linear Algebra for Machine Learning

In [1]:
import numpy as np
import pandas as pd

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

import ipywidgets as widgets
from ipywidgets import interact, interactive, fixed, interact_manual
from ipywidgets import IntSlider, FloatSlider, Dropdown, Button, Output, HBox, VBox, Label

from IPython.display import display, HTML, clear_output
from scipy import stats
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this demo you will be able to:

- **Trace** a complete forward pass through a neural network as a sequence of matrix multiplications
- **Identify** which linear algebra operation underpins each ML algorithm covered in the course
- **Interpret** a data science pipeline, from raw feature matrix to prediction, as one unified linear algebra computation

> **Tip:** Start by selecting **Linear Regression** from the algorithm dropdown and tracing the forward pass step by step. Then switch to **Neural Network** and count how many matrix multiplications are added with each hidden layer.

---
## How we got here

In *01–08* of the linear algebra series we built every tool individually: vectors, matrices, multiplication, transpose, inverse, norms, eigenvalues, and projections. This final notebook is a synthesis. Every concept from the series, and every concept from the calculus series before it, converges here into a single unified picture of how machine learning algorithms work as linear algebra computations.

---
## Why this matters for data science

When you call `model.fit(X, y)` in scikit-learn, you are triggering a sequence of matrix operations. When you debug a shape mismatch error in a neural network, you are reasoning about matrix dimensions. When you tune PCA, you are choosing how many eigenvectors to keep. The abstraction of "training a model" hides thousands of matrix multiplications per second. Seeing through that abstraction makes you a more capable practitioner: better at debugging, better at architecture choices, and better at understanding why algorithms behave the way they do.

---
## Try it yourself

In [2]:
def build_chain(algo, p, L, h):
    nodes = [dict(name="X", cols=p, kind="input")]
    edges = []
    if algo == "Linear Regression":
        edges.append(dict(op="× θ",
            line=f"(n,{p}) × ({p},) → (n,)", inner=p, matmul=True))
        nodes.append(dict(name="ŷ", cols=1, kind="output"))
    elif algo == "Logistic Regression":
        edges.append(dict(op="× θ, then σ",
            line=f"(n,{p}) × ({p},) → (n,)", inner=p, matmul=True))
        nodes.append(dict(name="ŷ", cols=1, kind="output"))
    else:
        prev = p
        for l in range(1, L + 1):
            edges.append(dict(op=f"× W{l}+b{l}, ReLU",
                line=f"(n,{prev}) × ({prev},{h}) → (n,{h})", inner=prev, matmul=True))
            nodes.append(dict(name=f"A{l}", cols=h, kind="hidden"))
            prev = h
        edges.append(dict(op=f"× W{L+1}+b{L+1}, σ",
            line=f"(n,{prev}) × ({prev},1) → (n,1)", inner=prev, matmul=True))
        nodes.append(dict(name="ŷ", cols=1, kind="output"))
    return nodes, edges

_KIND_COLORS = {
    "input":  PALETTE["muted"],
    "hidden": PALETTE["primary"],
    "output": PALETTE["secondary"],
}

out = Output()
_initialized = False

algo_dd = Dropdown(
    options=["Linear Regression", "Logistic Regression", "Neural Network"],
    value="Linear Regression",
    description="Algorithm:",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="380px"))
p_slider = IntSlider(value=8, min=2, max=16, step=1,
    description="Features p:",
    style={"description_width": "90px"},
    layout=widgets.Layout(width="380px"))
layers_slider = IntSlider(value=2, min=1, max=4, step=1,
    description="Hidden layers:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="380px"))
width_slider = IntSlider(value=6, min=2, max=12, step=1,
    description="Hidden width h:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="380px"))

def render(change=None):
    global _initialized
    algo = algo_dd.value
    p    = p_slider.value
    L    = layers_slider.value
    h    = width_slider.value

    nodes, edges = build_chain(algo, p, L, h)
    n_matmul = sum(e["matmul"] for e in edges)
    n_nodes  = len(nodes)

    if algo in ("Linear Regression", "Logistic Regression"):
        n_params = p + 1
    else:
        n_params = (p * h + h) + (L - 1) * (h * h + h) + (h * 1 + 1)

    max_cols = max(node["cols"] for node in nodes)

    def half_w(cols):
        return 0.10 + 0.30 * (cols / max_cols)

    fig = go.Figure()

    # Boxes
    for i, node in enumerate(nodes):
        w = half_w(node["cols"])
        fig.add_shape(type="rect",
            x0=i - w, x1=i + w, y0=-0.45, y1=0.45,
            fillcolor=_KIND_COLORS[node["kind"]],
            opacity=0.85,
            line=dict(color="white", width=2))
        fig.add_annotation(
            x=i, y=0,
            text=f"<b>{node['name']}</b><br>(n, {node['cols']})",
            showarrow=False,
            font=dict(color="white", size=11, family=FONT["family"]),
            align="center")

    # Arrows + labels
    for i, edge in enumerate(edges):
        x_tail = i + half_w(nodes[i]["cols"])
        x_head = (i + 1) - half_w(nodes[i + 1]["cols"])
        x_mid  = (x_tail + x_head) / 2

        fig.add_annotation(
            x=x_head, y=0,
            ax=x_tail, ay=0,
            xref="x", yref="y", axref="x", ayref="y",
            text="",
            showarrow=True,
            arrowhead=2, arrowsize=1.2, arrowwidth=2,
            arrowcolor="#888888")

        fig.add_annotation(
            x=x_mid, y=0.70,
            text=edge["op"],
            showarrow=False,
            font=dict(size=10, family=FONT["family"], color="#333"),
            align="center")

        inner_str = str(edge["inner"])
        shape_html = (
            edge["line"].replace(inner_str, f"<b>{inner_str}</b>") +
            "<br><span style='font-size:9px;color:#888'>(inner dims cancel)</span>")
        fig.add_annotation(
            x=x_mid, y=-0.72,
            text=shape_html,
            showarrow=False,
            font=dict(size=9, family=FONT["family"], color="#555"),
            align="center")

    plural_s = "s" if n_matmul != 1 else ""
    fig.update_layout(
        title=dict(
            text=(f"{algo}  |  {n_matmul} matrix multiplication{plural_s}  "
                  f"|  {n_params:,} parameters"),
            font=dict(size=FONT["size_title"], family=FONT["family"])),
        paper_bgcolor=PALETTE["background"],
        xaxis=dict(range=[-0.6, n_nodes - 0.4],
                   showgrid=False, showticklabels=False, zeroline=False),
        yaxis=dict(range=[-1.2, 1.2],
                   showgrid=False, showticklabels=False, zeroline=False),
        height=360,
        margin=dict(t=60, b=20, l=20, r=20),
        showlegend=False)

    # Interpretation
    line1 = (f"A forward pass in {algo} is {n_matmul} matrix multiplication{plural_s}. "
             f"Read left to right: each box is the data at that stage (n rows = your batch); "
             f"each arrow multiplies by a weight matrix.")
    line2 = ("At every multiply the inner dimensions must match — the columns of the left box "
             "equal the rows of the weight matrix and cancel, leaving the outer dimensions. "
             "A mismatch here is the shape error you debug in practice.")
    if algo == "Linear Regression":
        line3 = "No nonlinearity — the prediction is a single matrix–vector product Xθ."
    elif algo == "Logistic Regression":
        line3 = ("The same single product as linear regression, then a sigmoid squashes it "
                 "into (0,1) probabilities.")
    else:
        layer_s = "s" if L != 1 else ""
        relu_s  = "s" if L != 1 else ""
        line3 = (f"{L} hidden layer{layer_s} → {L+1} matmuls and {L} ReLU{relu_s} before "
                 f"the final sigmoid. Drag 'Hidden layers' up and watch each new layer add "
                 f"exactly one box, one matrix multiply, and one activation. "
                 f"(Features/width sliders reshape the boxes; layers/width apply only to the Neural Network.)")
    line4 = f"Total learnable parameters: {n_params:,}."

    interp = (
        f'<div style="font-family:{FONT["family"]};font-size:13px;background:#f8f9fa;'
        f'padding:14px 18px;border-radius:6px;margin-top:8px;line-height:1.65;color:#333;">'
        f'<p style="margin:0 0 6px 0;">{line1}</p>'
        f'<p style="margin:0 0 6px 0;">{line2}</p>'
        f'<p style="margin:0 0 6px 0;">{line3}</p>'
        f'<p style="margin:0;">{line4}</p>'
        f'</div>'
    )

    with out:
        clear_output(wait=True)
        fig.show()
        display(HTML(interp))
    _initialized = True

def on_change(change):
    render()

algo_dd.observe(on_change, names="value")
p_slider.observe(on_change, names="value")
layers_slider.observe(on_change, names="value")
width_slider.observe(on_change, names="value")

display(VBox([algo_dd, p_slider, HBox([layers_slider, width_slider]), out]))
render()

---
## What's happening?

Every supervised ML algorithm can be written as a sequence of linear algebra operations. The differences between algorithms come from: which operations are stacked, what non-linearities are added between them, and how the parameters are found.

**Linear Regression** prediction (one step):

$$\hat{y} = \mathbf{X} \boldsymbol{\theta}$$

where **X** is (n, p), **θ** is (p,), and ŷ is (n,). One matrix-vector multiplication.

**Logistic Regression** (one step + activation):

$$\hat{y} = \sigma(\mathbf{X} \boldsymbol{\theta})$$

Same multiplication, but passed through the sigmoid function to produce probabilities.

**Two-layer Neural Network** (three operations):

$$\mathbf{Z}_1 = \mathbf{X} \mathbf{W}_1 + \mathbf{b}_1$$
$$\mathbf{A}_1 = \text{ReLU}(\mathbf{Z}_1)$$
$$\hat{y} = \sigma(\mathbf{A}_1 \mathbf{W}_2 + \mathbf{b}_2)$$

Two matrix multiplications, two bias additions, two activation functions.

**PCA** (eigenvector decomposition + projection):

$$\mathbf{Z} = \mathbf{X} \mathbf{U}_k$$

where **U_k** contains the top k eigenvectors (columns). One matrix multiplication reduces 500 dimensions to k.

| ML algorithm | Core matrix operation | Linear algebra concept |
|-------------|----------------------|----------------------|
| Linear regression | Xθ = ŷ | Matrix-vector multiplication |
| Normal equations | θ = (XᵀX)⁻¹Xᵀy | Transpose, inverse, multiplication |
| Logistic regression | σ(Xθ) | Matrix multiply + element-wise function |
| Neural network layer | Z = XW + b | Matrix multiply + broadcast addition |
| PCA | Z = XU_k | Eigendecomposition + projection |
| KNN | argmin ‖x − xᵢ‖₂ | L2 norm, distance computation |
| SVM (linear) | sign(Xw + b) | Dot product, margin maximization |
| Ridge regression | minimize ‖Xθ − y‖₂² + λ‖θ‖₂² | L2 norm in objective + normal equations |
| Lasso | minimize ‖Xθ − y‖₂² + λ‖θ‖₁ | L1 norm in objective |

Return to the widget and step through the neural network forward pass. At each step, write down the input shape, the weight shape, and the output shape. After the final step, verify the total chain produces one prediction per example.

---
## Real-world example: A complete ML pipeline from raw data to prediction

The chart traces a realistic pipeline: feature matrix construction, standardization, PCA dimensionality reduction, and logistic regression prediction. Every step is a matrix operation.

Notice:

- **Notice:** The feature matrix X starts at shape (n, 500); after PCA it becomes (n, 20); every downstream operation now runs on 20 features instead of 500, which is 25x fewer multiplications per forward pass
- **Notice:** Standardization (subtracting the mean and dividing by standard deviation) is matrix subtraction and scalar division applied to entire columns, the broadcast operations from notebook 02 in action
- **Notice:** The logistic regression prediction is a single matrix-vector multiplication followed by sigmoid; the entire model, from raw features to probability, is a composition of linear algebra operations

> **Discussion question:** You are building a text classification model. Each document is a 10,000-dimensional TF-IDF vector. Your training set has 50,000 documents. Describe the shape of the feature matrix, the PCA weight matrix (if you reduce to 100 components), and the logistic regression weight vector. How many total floating-point numbers are involved in one prediction?

In [3]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification

np.random.seed(42)

X_raw, y = make_classification(n_samples=300, n_features=500,
                                n_informative=20, random_state=42)
n, p_raw = X_raw.shape

scaler = StandardScaler()
X_std  = scaler.fit_transform(X_raw)

pca    = PCA(n_components=20)
X_pca  = pca.fit_transform(X_std)

clf    = LogisticRegression(max_iter=1000).fit(X_pca, y)

_ACCENT = PALETTE.get("accent", PALETTE["secondary"])

pipeline_steps = [
    ("Raw data X",            (n, p_raw), PALETTE["muted"]),
    ("Standardized X_std",    (n, p_raw), PALETTE["primary"]),
    ("PCA Z = X U_k",         (n, 20),    PALETTE["secondary"]),
    ("Logistic Reg: Z·w",     (n,),       _ACCENT),
    ("sigmoid(Z·w) = ŷ",      (n,),       _ACCENT),
]

labels  = [s[0] for s in pipeline_steps]
heights = [s[1][0] * (s[1][1] if len(s[1]) > 1 else 1) for s in pipeline_steps]
colors  = [s[2] for s in pipeline_steps]
shapes  = [str(s[1]) for s in pipeline_steps]

fig = go.Figure(go.Bar(
    x=labels, y=heights,
    marker_color=colors,
    text=shapes, textposition="outside",
))
fig.update_layout(
    title=dict(
        text="ML Pipeline as Matrix Operations — Shape Shrinks from (300, 500) to (300,) via PCA",
        font=dict(size=FONT["size_title"], family=FONT["family"])),
    xaxis_title="Pipeline Step",
    yaxis_title="Total Elements (shape product)",
    paper_bgcolor=PALETTE["background"],
    plot_bgcolor=PALETTE["background"],
    height=420,
    margin=dict(t=60, b=40, l=70, r=20),
)
fig.update_xaxes(autorange=True)
fig.update_yaxes(autorange=True)
display(go.FigureWidget(fig))

FigureWidget({
    'data': [{'marker': {'color': ['#868E96', '#4C6EF5', '#F76707', '#2F9E44', '#2F9E44']},
              'text': [(300, 500), (300, 500), (300, 20), (300,), (300,)],
              'textposition': 'outside',
              'type': 'bar',
              'uid': '7c5262ac-667e-44f2-ba84-be9dfd2f0747',
              'x': [Raw data X, Standardized X_std, PCA Z = X U_k, Logistic Reg:
                    Z·w, sigmoid(Z·w) = ŷ],
              'y': [150000, 150000, 6000, 300, 300]}],
    'layout': {'height': 420,
               'margin': {'b': 40, 'l': 70, 'r': 20, 't': 60},
               'paper_bgcolor': '#F8F9FA',
               'plot_bgcolor': '#F8F9FA',
               'template': '...',
               'title': {'font': {'family': 'Inter, Arial, sans-serif', 'size': 18},
                         'text': ('ML Pipeline as Matrix Operatio' ... 'm (300, 500) to (300,) via PCA')},
               'xaxis': {'autorange': True, 'title': {'text': 'Pipeline Step'}},
               'yaxis'

### Linear algebra operations in ML: complete reference

| ML algorithm | Core matrix operation | numpy/sklearn method | What the numbers represent |
|-------------|----------------------|---------------------|--------------------------|
| Linear regression | Xθ = ŷ | `X @ theta` | Feature values weighted and summed |
| Normal equations | (XᵀX)⁻¹Xᵀy | `np.linalg.solve(X.T@X, X.T@y)` | Exact optimal weights |
| Neural net layer | XW + b | `X @ W + b` | Weighted combination of inputs + bias |
| PCA | XU_k | `pca.transform(X)` | Coordinates along principal components |
| KNN distance | ‖x − xᵢ‖₂ | `np.linalg.norm(x - xi)` | Euclidean distance to each neighbor |
| Ridge regularization | ‖θ‖₂² | `np.sum(theta**2)` | Penalty on large weights |
| Lasso regularization | ‖θ‖₁ | `np.sum(np.abs(theta))` | Penalty that encourages zero weights |
| Backpropagation | Wᵀ δ | `W.T @ delta` | Gradient flowing backward through layer |

---
## Key takeaway

> **Every machine learning algorithm is linear algebra: data is a matrix, parameters are vectors or matrices, predictions are matrix multiplications, regularization is a norm, and dimensionality reduction is a projection — understanding these operations is understanding the models themselves.**

---
*You have completed the linear algebra series. Together with the calculus series, you now have the mathematical foundation for every algorithm covered in the remainder of this course.*